In [31]:
import feedparser
import time
import re # Sử dụng Regular Expression để bắt từ chính xác

FEEDS = [
    "https://www.lecho.be/rss/top_stories.xml",  
    "https://www.lecho.be/rss/entreprises_banques.xml"
]

WATCHLIST = [ "bnp paribas", "fortis", "kbc", "belfius", "argenta", "crelan", 
              "beobank", "vdk", "nagelmackers", "axa", "fintro", "deutsche bank", 
              "keytrade", "medirect", "santander", "nibc", "bunq", "hello bank", 
              "izola", "cph", "bankb", "triodos", "europabank", "unicredit",
              "épargne", "placement", "investissement", "banque", "ing", "taux"]

def monitor_news():
    print(f"--- ING DAILY WATCH TOOL: {time.ctime()} ---")
    seen_articles = set()

    for url in FEEDS:
        feed = feedparser.parse(url)
        for entry in feed.entries:
            title = entry.title.lower()
            summary = getattr(entry, 'summary', "").lower()
            full_text = title + " " + summary

            # SỬA LỖI Ở ĐÂY: Sử dụng Regex để tìm từ nguyên vẹn
            # \b giúp xác định ranh giới từ (không bắt 'ing' trong 'shopping')
            found = False
            for key in WATCHLIST:
                if re.search(r'\b' + re.escape(key) + r'\b', full_text):
                    found = True
                    break
            
            if found:
                if entry.link not in seen_articles:
                    print(f"\n[MATCH FOUND] - Keyword detected")
                    print(f"Title: {entry.title}")
                    print(f"Link:  {entry.link}")
                    print(f"Date:  {entry.published}")
                    seen_articles.add(entry.link)

if __name__ == "__main__":
    monitor_news()

--- ING DAILY WATCH TOOL: Tue May  5 13:34:06 2026 ---

[MATCH FOUND] - Keyword detected
Title: BNP Paribas Fortis lance un nouveau compte d'épargne à 2,9%, mais sous conditions
Link:  https://www.lecho.be/r/t/1/id/10659036
Date:  Mon, 04 May 2026 09:33:29 GMT

[MATCH FOUND] - Keyword detected
Title: Marc Raisière (Belfius): "En Belgique, on a encore du mal à être fiers de notre succès"
Link:  https://www.lecho.be/r/t/1/id/10658464
Date:  Sat, 02 May 2026 02:56:14 GMT

[MATCH FOUND] - Keyword detected
Title: ING va racheter 1 milliard d'euros d'actions propres en plus après un trimestre solide
Link:  https://www.lecho.be/r/t/1/id/10658658
Date:  Thu, 30 Apr 2026 07:02:12 GMT

[MATCH FOUND] - Keyword detected
Title: Nouveau record de bénéfices pour BNP Paribas, malgré les tensions géopolitiques
Link:  https://www.lecho.be/r/t/1/id/10658648
Date:  Thu, 30 Apr 2026 06:30:27 GMT

[MATCH FOUND] - Keyword detected
Title: Les apps d'investissement belges restent parmi les plus chères du march

In [33]:
import feedparser
import json
import hashlib
import re
import datetime
from pathlib import Path

# Configuration
SNAPSHOT_DIR = Path("snapshots")
FEEDS = {
    "top_stories": "https://www.lecho.be/rss/top_stories.xml",
    "entreprises_banques": "https://www.lecho.be/rss/entreprises_banques.xml"
}

WATCHLIST = [
    "bnp paribas", "fortis", "kbc", "belfius", "argenta", "crelan", 
    "beobank", "vdk", "nagelmackers", "axa", "fintro", "deutsche bank", 
    "keytrade", "medirect", "santander", "nibc", "bunq", "hello bank", 
    "izola", "cph", "bankb", "triodos", "europabank", "unicredit",
    "épargne", "placement", "investissement", "banque", "ing", "taux"
]

def compute_checksum(data: str) -> str:
    """Compute SHA-256 checksum of the content for change detection."""
    return hashlib.sha256(data.encode("utf-8")).hexdigest()

def build_snapshot(feed_name: str, url: str, articles: list[dict]) -> dict:
    """
    Build the final structured snapshot with metadata.
    Standardized output format for the pipeline.
    """
    return {
        "source": "lecho.be",
        "feed": feed_name,
        "url": url,
        "timestamp": datetime.datetime.now().isoformat(),
        "date": datetime.date.today().isoformat(),
        "num_articles": len(articles),
        "checksum": compute_checksum(json.dumps(articles, sort_keys=True)),
        "articles": articles,
    }

def save_snapshot(snapshot: dict, date_str: str, feed_name: str) -> Path:
    """Save structured snapshot as JSON."""
    SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
    # Changed name from bankshopper to lecho
    filepath = SNAPSHOT_DIR / f"lecho_{feed_name}_{date_str}.json"
    filepath.write_text(
        json.dumps(snapshot, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    return filepath

def run_monitor():
    date_str = datetime.date.today().isoformat()
    
    for feed_name, url in FEEDS.items():
        feed = feedparser.parse(url)
        filtered_articles = []

        for entry in feed.entries:
            title = entry.title
            description = getattr(entry, 'summary', "")
            content_to_scan = (title + " " + description).lower()

            # Exact word boundary check to avoid 'shopping' matching 'ing'
            is_match = any(re.search(r'\b' + re.escape(key) + r'\b', content_to_scan) for key in WATCHLIST)
            
            if is_match:
                filtered_articles.append({
                    "guid": getattr(entry, 'id', entry.link),
                    "title": title,
                    "description": description,
                    "link": entry.link,
                    "pub_date": getattr(entry, 'published', ""),
                    "category": getattr(entry, 'category', "News")
                })

        # Build the standardized snapshot
        snapshot = build_snapshot(feed_name, url, filtered_articles)
        
        # Save to disk
        saved_path = save_snapshot(snapshot, date_str, feed_name)
        print(f"Successfully saved {feed_name} snapshot to: {saved_path}")

if __name__ == "__main__":
    run_monitor()

Successfully saved top_stories snapshot to: snapshots/lecho_top_stories_2026-05-05.json
Successfully saved entreprises_banques snapshot to: snapshots/lecho_entreprises_banques_2026-05-05.json
